# 03 - Análise de Fluxos (Captação/Resgate)

Análise de movimentações (captação e resgate) dos fundos REAG para identificar padrões suspeitos.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.processors.data_processor import DataProcessor
from src.analyzers.anomaly_detector import AnomalyDetector
from config.settings import Config

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

## Carregar Lista de Fundos REAG

In [2]:
config = Config()
processor = DataProcessor(config)
detector = AnomalyDetector(config)

# Carregar lista de fundos REAG
reag_list_path = config.PROCESSED_DATA_DIR / 'reag_fund_list.csv'

if reag_list_path.exists():
    df_reag_list = pd.read_csv(reag_list_path)
    reag_cnpjs = df_reag_list['CNPJ_FUNDO'].tolist()
    print(f"✅ {len(reag_cnpjs)} fundos REAG carregados")
else:
    print("⚠️ Execute primeiro o notebook 02_identify_reag_funds.ipynb")
    reag_cnpjs = []

⚠️ Execute primeiro o notebook 02_identify_reag_funds.ipynb


## Carregar e Consolidar Informe Diário

In [3]:
# Ler todos os arquivos de Informe Diário
# CVM files are named: inf_diario_fi_YYYYMM.csv
informe_files = sorted(config.RAW_DATA_DIR.glob('inf_diario_fi_*.csv'))

# Also check in informe subdirectory if it exists
informe_dir = config.RAW_DATA_DIR / 'informe'
if informe_dir.exists():
    informe_files.extend(list(informe_dir.glob('inf_diario_fi_*.csv')))

informe_files = sorted(set(informe_files))

print(f"📂 Encontrados {len(informe_files)} arquivos de Informe Diário")

if not informe_files:
    print("⚠️ Nenhum arquivo de Informe Diário encontrado")
    print(f"   Procurado em: {config.RAW_DATA_DIR}")
    print(f"   Padrão: inf_diario_fi_*.csv")
    print("\n💡 Execute o notebook 01_data_collection.ipynb para baixar os dados")
    df_informe = pd.DataFrame()
else:
    dfs = []
    for file in informe_files:
        print(f"  Lendo {file.name}...")
        df = processor.read_informe_diario(file)
        dfs.append(df)

    df_informe = pd.concat(dfs, ignore_index=True)
    print(f"\n📊 Total de registros: {len(df_informe):,}")

📂 Encontrados 0 arquivos de Informe Diário
⚠️ Nenhum arquivo de Informe Diário encontrado
   Procurado em: /Users/pedrotodescan/Documents/Dev/REAG/notebooks/../data/raw
   Padrão: inf_diario_fi_*.csv

💡 Execute o notebook 01_data_collection.ipynb para baixar os dados


## Filtrar Fundos REAG

In [4]:
df_reag = processor.filter_by_cnpj(df_informe, reag_cnpjs)
print(f"📊 Registros de fundos REAG: {len(df_reag):,}")

if df_reag.empty:
    print("⚠️ Nenhum registro REAG encontrado. Verifique os dados do Informe Diário e a lista de fundos.")
else:
    # Calcular fluxo líquido
    df_reag = processor.calculate_net_flow(df_reag)

    # Ordenar por data
    df_reag = df_reag.sort_values(['CNPJ_FUNDO', 'DT_COMPTC'])


📊 Registros de fundos REAG: 0
⚠️ Nenhum registro REAG encontrado. Verifique os dados do Informe Diário e a lista de fundos.


## Estatísticas Descritivas

In [5]:
print("")
print("📊 ESTATÍSTICAS DE FLUXO")
print("="*60)

if df_reag.empty:
    print("⚠️ Sem dados para estatísticas.")
    total_captacao = 0
    total_resgate = 0
    fluxo_liquido_total = 0
else:
    # Agregado total
    total_captacao = df_reag['CAPTC_DIA'].sum()
    total_resgate = df_reag['RESG_DIA'].sum()
    fluxo_liquido_total = df_reag['FLUXO_LIQ_DIA'].sum()

    print(f"Captação total: R$ {total_captacao:,.2f}")
    print(f"Resgate total: R$ {total_resgate:,.2f}")
    print(f"Fluxo líquido total: R$ {fluxo_liquido_total:,.2f}")

    print("")
    print("📈 Estatísticas descritivas:")
    display(df_reag[['CAPTC_DIA', 'RESG_DIA', 'FLUXO_LIQ_DIA', 'VL_PATRIM_LIQ']].describe())



📊 ESTATÍSTICAS DE FLUXO
⚠️ Sem dados para estatísticas.


## Evolução Temporal do Fluxo

In [6]:
# Agregar por data
if df_reag.empty:
    print("⚠️ Sem dados para agregação diária.")
    df_daily = pd.DataFrame()
else:
    df_daily = df_reag.groupby('DT_COMPTC').agg({
        'CAPTC_DIA': 'sum',
        'RESG_DIA': 'sum',
        'FLUXO_LIQ_DIA': 'sum',
        'VL_PATRIM_LIQ': 'sum'
    }).reset_index()

# Plot
if not df_daily.empty:
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))

    # Fluxo líquido
    axes[0].plot(df_daily['DT_COMPTC'], df_daily['FLUXO_LIQ_DIA'], label='Fluxo Líquido', linewidth=2)
    axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
    axes[0].set_title('Fluxo Líquido Diário - Fundos REAG', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('R$ Milhões')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # PL agregado
    axes[1].plot(df_daily['DT_COMPTC'], df_daily['VL_PATRIM_LIQ'], label='PL Total', linewidth=2, color='green')
    axes[1].set_title('Patrimônio Líquido Total - Fundos REAG', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('R$ Milhões')
    axes[1].set_xlabel('Data')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


⚠️ Sem dados para agregação diária.


## Top Fundos por Volume de Movimentação

In [7]:
# Agregado por fundo
if df_reag.empty:
    print("⚠️ Sem dados para agregação por fundo.")
    df_by_fund = pd.DataFrame()
else:
    df_by_fund = df_reag.groupby('CNPJ_FUNDO').agg({
        'CAPTC_DIA': 'sum',
        'RESG_DIA': 'sum',
        'FLUXO_LIQ_DIA': 'sum',
        'VL_PATRIM_LIQ': 'last'
    }).reset_index()

    df_by_fund['VOLUME_TOTAL'] = df_by_fund['CAPTC_DIA'] + df_by_fund['RESG_DIA']

    # Top 10 por volume
    top_funds = df_by_fund.nlargest(10, 'VOLUME_TOTAL')

    print("")
    print("🏆 TOP 10 FUNDOS POR VOLUME DE MOVIMENTAÇÃO")
    print("="*60)
    display(top_funds)


⚠️ Sem dados para agregação por fundo.


## Salvar Dados Processados

In [8]:
# Salvar dados REAG processados
if df_reag.empty:
    print("⚠️ Sem dados para salvar.")
else:
    output_path = processor.save_processed(df_reag, 'reag_informe_diario_processed.csv')
    print("")
    print(f"✅ Dados salvos em: {output_path}")

    # Salvar agregados
    if not df_by_fund.empty:
        output_by_fund = processor.save_processed(df_by_fund, 'reag_summary_by_fund.csv')
        print(f"✅ Resumo por fundo salvo em: {output_by_fund}")


⚠️ Sem dados para salvar.


## Resumo

In [9]:
print("")
print("="*60)
print("📊 RESUMO DA ANÁLISE DE FLUXOS")
print("="*60)
if df_reag.empty:
    print("⚠️ Nenhum dado disponível para o resumo.")
else:
    print(f"Fundos analisados: {df_reag['CNPJ_FUNDO'].nunique()}")
    print(f"Período: {df_reag['DT_COMPTC'].min()} a {df_reag['DT_COMPTC'].max()}")
    print(f"Captação total: R$ {total_captacao:,.2f}")
    print(f"Resgate total: R$ {total_resgate:,.2f}")
    print(f"Fluxo líquido: R$ {fluxo_liquido_total:,.2f}")
    print("")
    print("✅ Análise concluída! Próximo passo: 04_anomaly_detection.ipynb")



📊 RESUMO DA ANÁLISE DE FLUXOS
⚠️ Nenhum dado disponível para o resumo.
